In [12]:
import pickle
import pandas as pd
import numpy as np
from pypcd4 import PointCloud
import cv2
from PIL import Image
import json
import math
import os

In [6]:
# Необходимые столбцы
CUBOIDS_COLUMNS = ['label', 'yaw', 'position.x', 'position.y', 'dimensions.x', 'dimensions.y']
pcd_columns = ['x', 'y', 'z', 'i']
BEV_LABELS = ['Car', 'Bus', 'Pickup Truck']

# Квадрат для препроцессинга
PCD_AREA_WIDTH = 50
PCD_AREA_HEIGHT = 50

# Размер изображения после препроцессинга
BEV_WIDTH = 640
BEV_HEIGHT = 640

# Ограничения по дальности лидара
WIDTH_MAGIC = 50
HEIGHT_MAGIC = 50

In [ ]:
cur_folder = '047'
cur_file = '01.pkl'

with open(f'./training/Pandaset/{cur_folder}/annotations/cuboids/{cur_file}', 'rb') as f:

    cuboids_data = pickle.load(f)
    # print(set(cuboids_data['cuboids.sibling_id']))
    # print(cuboids_data.columns)
    # cuboids_data = cuboids_data[cuboids_data['cuboids.sibling_id'].isin(['6c71e7b5-8eb3-4d3c-876f-e258e0e399db', '52195828-398a-4f69-a0ee-7ba9ce260ee1', '64b6e696-e63f-406c-b5f3-a8ca3668de3d', 'e5ad8a8c-8a62-4e7e-a08b-3a8a51ba0fd6',])]
    cuboids_data = cuboids_data[cuboids_data['cuboids.sensor_id'].isin([-1, 0])]
    cuboids_data = cuboids_data[CUBOIDS_COLUMNS]
    cuboids_data = cuboids_data[cuboids_data['label'].isin(BEV_LABELS)]
    cuboids_array = cuboids_data.to_numpy()

with open(f'./training/Pandaset/{cur_folder}/lidar/{cur_file}', 'rb') as f:
    lidar_data = pickle.load(f)
    points_array = lidar_data[pcd_columns].to_numpy()
    

/tmp/ipykernel_5550/1464613782.py:6: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  cuboids_data = pickle.load(f)
/tmp/ipykernel_5550/1464613782.py:16: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  lidar_data = pickle.load(f)


In [24]:
with open(f'training/Pandaset/{cur_folder}/lidar/poses.json') as f:
    json_poses = json.load(f)

files = os.listdir(f"./training/Pandaset/{cur_folder}/lidar")
files.sort()
file_idx= files.index(cur_file)

position = json_poses[file_idx]['position']
x_center = position['x']
y_center = position['y']

In [25]:
def xywhr_to_4xy(xywhr):
    cy, cx, w, h, r = xywhr
    yaw = r - np.pi / 2
    
    cos_a = np.cos(yaw)
    sin_a = np.sin(yaw)
    
    R = np.array([[cos_a, -sin_a], 
                  [sin_a,  cos_a]])
    
    dx, dy = w / 2, h / 2
    corners = np.array([
        [-dx,  dy],
        [ dx,  dy],
        [-dx, -dy],
        [ dx, -dy]
    ])
    
    points = np.round(corners @ R + [cx, cy], 4)
    
    return points.flatten()

def get_label_index(label):
    label_index = BEV_LABELS.index(label)
    return label_index

def get_normalized_cuboid(cuboid_data, row, col, x_min_border, y_min_border):
    x_min_area_border = x_min_border + row * PCD_AREA_WIDTH
    y_min_area_border = y_min_border + col * PCD_AREA_HEIGHT
    
    normalized_cuboid = cuboid_data.copy()
    
    normalized_cuboid[2] = (np.float32(normalized_cuboid[2]) - x_min_area_border) / PCD_AREA_WIDTH
    normalized_cuboid[3] = (np.float32(normalized_cuboid[3]) - y_min_area_border) / PCD_AREA_HEIGHT
    
    normalized_cuboid[4] = np.float32(normalized_cuboid[4]) / PCD_AREA_WIDTH
    normalized_cuboid[5] = np.float32(normalized_cuboid[5]) / PCD_AREA_HEIGHT
    
    return normalized_cuboid


def get_labels(cuboid_data):
    label_data = np.zeros(9)

    label_data[0] = get_label_index(cuboid_data[0])

    xywhr = np.zeros(5)
    xywhr[:4] = cuboid_data[2:6]
    xywhr[4] = cuboid_data[1] 

    cuboid_points = xywhr_to_4xy(xywhr)
    label_data[1:] = cuboid_points

    return label_data

In [28]:
x_min_border = x_center - WIDTH_MAGIC
x_max_border = x_center + WIDTH_MAGIC

y_min_border = y_center - HEIGHT_MAGIC
y_max_border = y_center + HEIGHT_MAGIC

print(f"Зона для обучения: [({x_min_border}, {y_min_border}) : ({x_max_border}, {y_max_border})]")

# y (cols)
# |
# |
# 0 --- x  (rows)

row_num = int((x_max_border - x_min_border - 1) // PCD_AREA_WIDTH + 1)
col_num = int((y_max_border - y_min_border - 1) // PCD_AREA_HEIGHT + 1)
print(f"{row_num}x{col_num} зон для обучения")

# Для нормализации z
z_min = np.min(points_array[:, 2])
z_max = np.max(points_array[:, 2])

Зона для обучения: [(-49.89242766591586, -50.604309930436614) : (50.10757233408414, 49.395690069563386)]
2x2 зон для обучения


In [27]:
# Распределение всех точек по необходимым зонам (row_num x col_num)
def get_point_areas(points_array, row_num, col_num, x_min_border, y_min_border):
    point_areas = [[[] for _ in range((col_num))] for _ in range(row_num)]

    for point in points_array:
        
        x_idx = int((point[0] - x_min_border) // PCD_AREA_WIDTH)
        y_idx = int((point[1] - y_min_border) // PCD_AREA_HEIGHT)
        if x_idx >= 0 and y_idx >= 0 and x_idx < row_num and y_idx < col_num:
            point_areas[x_idx][y_idx].append(point)
    
    return point_areas

def get_label_areas(cuboids_array, row_num, col_num, x_min_border, y_min_border):
    label_areas = [[[] for _ in range((col_num))] for _ in range(row_num)]

    for cuboid in cuboids_array:
        x_idx = int((cuboid[2] - x_min_border) // PCD_AREA_WIDTH)
        y_idx = int((cuboid[3] - y_min_border) // PCD_AREA_HEIGHT)
        if x_idx >= 0 and y_idx >= 0 and x_idx < row_num and y_idx < col_num:
            normalize_cuboid = get_normalized_cuboid(cuboid, x_idx, y_idx, x_min_border, y_min_border)
            cuboid_labels = get_labels(normalize_cuboid)
            if np.min(cuboid_labels[1:]) < 0 or np.max(cuboid_labels[1:]) > 1:
                continue
            label_areas[x_idx][y_idx].append(cuboid_labels)
    
    return label_areas

point_areas = get_point_areas(points_array, row_num, col_num, x_min_border, y_min_border)
label_areas = get_label_areas(cuboids_array, row_num, col_num, x_min_border, y_min_border)



In [53]:
# Create (x, y): (counts, maxZ) dict
def get_CoordToCountValInt_dict(points):
    coord_to_countval = dict()
    for point in points:
        if coord_to_countval.get((point[0], point[1])) == None:
            coord_to_countval[((point[0], point[1]))] = [1, point[2], point[3]]
        else:
            coord_to_countval[((point[0], point[1]))][0] += 1
    return coord_to_countval


def pcd_to_img_map(points_array, x_idx, y_idx, x_min_border, y_min_border, z_min, z_max):
    x_min_area_border = x_min_border + x_idx * PCD_AREA_WIDTH
    y_min_area_border = y_min_border + y_idx * PCD_AREA_HEIGHT

    # Нормализуем x,y [0: 1]
    points_array[:, 0] = (points_array[:, 0] - x_min_area_border) / PCD_AREA_WIDTH
    points_array[:, 1] = (points_array[:, 1] - y_min_area_border) / PCD_AREA_HEIGHT

    # Приводим x,y к [0: BEV_H/BEV_W]
    points_array[:, 0] = np.int32(points_array[:, 0] * BEV_WIDTH)
    points_array[:, 1] = np.int32(points_array[:, 1] * BEV_HEIGHT)

    # Приводим z к [0, 1]
    points_array[:, 2] = (points_array[:, 2] - z_min) / (z_max - z_min)

    # Сортируем (увел х, увел y, умен z)
    ix = np.lexsort((-points_array[:, 2], points_array[:, 1], points_array[:, 0]))
    points_array = points_array[ix]

    height_map = np.zeros((BEV_HEIGHT, BEV_WIDTH))
    density_map = np.zeros((BEV_HEIGHT, BEV_WIDTH))
    intensity_map = np.zeros((BEV_HEIGHT, BEV_WIDTH))

    points_array[:, 0] = np.minimum(np.maximum(points_array[:, 0], 0), BEV_HEIGHT - 1)
    points_array[:, 1] = np.minimum(np.maximum(points_array[:, 1], 0), BEV_WIDTH - 1)

    coord_to_countval = get_CoordToCountValInt_dict(points_array)
    # points_array = points_array[ix]

    for ((x, y), (c, z, i)) in coord_to_countval.items():
        density_map[int(x)][int(y)] = min(1.0, np.log(c + 1) / np.log(64))
        height_map[int(x)][int(y)] = z
        intensity_map[int(x)][int(y)] = i

    img_map = np.zeros([BEV_HEIGHT, BEV_WIDTH, 3])
    img_map[:,:,0] = density_map
    img_map[:,:,1] = height_map
    # img_map[:,:,2] = height_map
    # img_map[:,:,2] = np.full([BEV_HEIGHT, BEV_WIDTH ], 1)
    img_map[:,:,2] = intensity_map
    
    return img_map

# temp = np.asarray(point_areas[0][0])
# img = pcd_to_img_map(temp, 0, 0, x_min_border, y_min_border, z_min, z_max)

# bevImage = img * 255

# cv2.imshow("BEV", img)
# cv2.waitKey(0)
# cv2.destroyAllWindows()
# cv2.imwrite("test.png", bevImage.astype(np.uint8))
# img = Image.fromarray(bevImage.astype(np.uint8))
# img

In [ ]:
def filter_empty_obbs(image, obbs, min_useful_ratio=0.03, black_thresh=10):
    valid_indices = []

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        
    for idx, obb in enumerate(obbs):
        pts = obb[1:].reshape(4, 2)
        
        pts_abs = pts * np.array([BEV_WIDTH, BEV_HEIGHT])
        
        pts_abs = np.round(pts_abs).astype(np.int32)
        
        x, y, w, h = cv2.boundingRect(pts_abs)
        
        x_min, y_min = max(0, x), max(0, y)
        x_max, y_max = min(BEV_WIDTH, x + w), min(BEV_HEIGHT, y + h)
        
        if x_max <= x_min or y_max <= y_min:
            continue
        
        roi_gray = gray[y_min:y_max, x_min:x_max]
        
        local_pts = pts_abs - [x_min, y_min]
        
        mask = np.zeros(roi_gray.shape, dtype=np.uint8)
        cv2.fillPoly(mask, [local_pts], 255)
        
        polygon_pixels = roi_gray[mask == 255]
        
        if len(polygon_pixels) == 0:
            continue
            
        useful_pixels_count = np.sum(polygon_pixels > black_thresh)
        useful_ratio = useful_pixels_count / len(polygon_pixels)
        
        if useful_ratio >= min_useful_ratio:
            valid_indices.append(idx)
            
    return obbs[valid_indices]

In [55]:
labels = []

for x in range(0, row_num):
    for y in range(0, col_num):
        cur_area_point_arr = np.asarray(point_areas[x][y])
        cur_area_label_arr = np.asarray(label_areas[x][y])

        if len(cur_area_label_arr) * len(cur_area_point_arr) == 0:
            continue
        
        img = pcd_to_img_map(cur_area_point_arr, x, y, x_min_border, y_min_border, z_min, z_max)
        bevImage = (img * 255).astype(np.uint8)
    
        area_labels = cur_area_label_arr.copy()
        area_labels = filter_empty_obbs(bevImage, area_labels)
        area_labels = area_labels[:, 1:] * 640
        
        if len(area_labels) == 0:
            continue
        
        labels.append(cur_area_label_arr)
        
        for cuboid in area_labels:
            cnt = np.array([cuboid[0:2], cuboid[4:6], cuboid[6:8], cuboid[2:4]], dtype=np.int32)
            # print(points)
            cv2.drawContours(img, [cnt], -1, (0, 255, 0), 3)
        cv2.imshow("Contour by 4 points", img)
        cv2.waitKey(0)
        cv2.destroyAllWindows()
            
        cv2.imwrite(f"test{x}{y}.png", bevImage.astype(np.uint8))


In [389]:
def labels_to_text(labels):
    label_text = ''
    for label in labels:
        label_text += f"{int(label[0])} {" ".join(label[1:].astype(str))}\n"
    return label_text

def write_labels(all_labels, labels_path, pandaset_folder_num, file):
    for idx in range(len(all_labels)):
        labels = all_labels[idx]
        label_text = labels_to_text(labels)
        # print(label_text)
        
        write_path = f"{labels_path}/{pandaset_folder_num}_{file[:-4]}_{idx:02d}.txt"
        with open(write_path, 'w') as labelfile:
            labelfile.write(label_text)
    


BEV_DATASET_PATH = './training/BEV_Dataset'
labels_path = f"{BEV_DATASET_PATH}/labels"
# labels = np.asarray(label_areas).flatten()
# print(len(labels))
write_labels(labels, labels_path, "001", "01")

In [328]:
import json
from datetime import datetime
import math


def poses_to_rads(poses_data):
    rads = []
    for pose_data in poses_data:
        yaw = get_z_rotation(pose_data['heading'])
        rads.append(yaw)
    
    return np.asarray(rads)

    

def get_z_rotation(heading):
    w = heading['w']
    x = heading['x']
    y = heading['y']
    z = heading['z']
    vx = 1 - 2 * (y**2 + z**2)
    vy = 2 * (x * y + w * z)
    angle_rad = math.atan2(vy, vx)
    # angle_deg = math.degrees(angle_rad)
    
    return angle_rad


my_test_vals = []

filenames = ['001', '002', '004', '008', '020', '030', '040', '041']

for filename in filenames:
    with open(f"./training/Pandaset/{filename}/lidar/poses.json", 'r') as file:
        poses_data = json.load(file)
        my_test_vals.append(poses_data[0])

yaw_all = [get_z_rotation(x['heading']) for x in my_test_vals]        
# print(my_test_vals)
# z1 = get_z_rotation(my_test_vals[0]['heading'])
# z2 = get_z_rotation(my_test_vals[1]['heading'])
# z3 = get_z_rotation(my_test_vals[2]['heading'])

print(yaw_all)

# print(z1 + np.pi / 4)
# print(f'{abs(z1 - z2)}::::::{np.pi / 4}')
# print(f'{abs(z1 - z3)}::::::{5 * np.pi / 12}')

# print(poses_to_rads(poses_data))

[-0.7944430034154631, -0.7903721575593857, -0.806450082947037, -0.7964144267351548, 0.6322791384360434, -1.3813610438660806, -2.998616796386284, -2.9816808561253767]
